# Annotation Precision Outliers
Interaktive Inspektion von Bildern mit hoher Inter-Rater-Abweichung.

**Vorbedingung:** `python prepare_outlier_data.py --token YOUR_TOKEN` einmalig ausführen.
Die Google-signierten Bild-URLs in `data/img_urls.pkl` laufen nach ~24 h ab.

In [1]:
import io
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image

from bspline_python import interpolate_bspline, spine_labels

%matplotlib inline
%load_ext autoreload
%autoreload 2

# Konva stage height (fixed in xray-annotator/src/components/KonvaImage.vue).
# All node/cobbLine coordinates are stored in this scaled space.
KONVA_STAGE_HEIGHT = 600

## Daten laden

In [2]:
df4d = pd.read_pickle('data/df4d_corrected_spine_measurements.pkl')
angles_df = pd.read_pickle('data/outlier_angles_df.pkl')
outlier_df = pd.read_pickle('data/outlier_ranking_df.pkl')
with open('data/img_urls.pkl', 'rb') as fh:
    img_urls = pickle.load(fh)

print(f'df4d:       {len(df4d)} Messungen, {df4d["id"].nunique()} Bilder')
print(f'angles_df:  {len(angles_df)} Zeilen, {angles_df["cobbType"].nunique()} cobbTypes')
print(f'outlier_df: {len(outlier_df)} Gruppen (≥2 Rater)')
print(f'img_urls:   {len(img_urls)} URLs')
print()
print('Verfügbare cobbTypes (mit ≥1 Gruppe in outlier_df):')
for ct in sorted(angles_df['cobbType'].unique()):
    n = (outlier_df['cobbType'] == ct).sum()
    if n > 0:
        print(f'  {ct:40s}  {n:4d} Gruppen')

df4d:       1137 Messungen, 503 Bilder
angles_df:  24046 Zeilen, 33 cobbTypes
outlier_df: 5668 Gruppen (≥2 Rater)
img_urls:   503 URLs

Verfügbare cobbTypes (mit ≥1 Gruppe in outlier_df):
  Cobb-regional                              205 Gruppen
  Cobb-segmental                            1276 Gruppen
  Spline                                     354 Gruppen
  inflection_r0@10                           147 Gruppen
  inflection_r0@15                           168 Gruppen
  inflection_r0@5                            134 Gruppen
  inflection_r1@10                           171 Gruppen
  inflection_r1@15                           135 Gruppen
  inflection_r1@5                            177 Gruppen
  inflection_r2@10                            70 Gruppen
  inflection_r2@15                            41 Gruppen
  inflection_r2@5                            117 Gruppen
  inflection_r3@10                            22 Gruppen
  inflection_r3@15                            10 Gruppen
  inflection_r

## Interaktiver Outlier-Viewer

In [3]:
class OutlierViewer:
    """Zeigt Top-N Outlier-Bilder für einen cobbType mit korrektem Koordinaten-Overlay.

    Koordinaten-Korrektur:
        Der xray-annotator skaliert jedes Bild auf eine feste Konva-Stage-Höhe von 600px.
        Alle gespeicherten Node- und CobbLine-Koordinaten sind in diesem Canvas-Raum.
        Beim Overlay auf das Original-Bild werden sie mit scale = img.height / 600 skaliert.
    """

    _COLORS = plt.cm.tab10.colors
    # cobbTypes für die CobbLines aus stateObject.cobbLines gezeichnet werden
    _COBB_LINE_TYPES = {'Cobb-regional', 'Cobb-segmental'}

    def __init__(self, df4d, outlier_df, angles_df, img_urls):
        self._df4d = df4d
        self._outlier = outlier_df
        self._angles = angles_df
        self._urls = img_urls

        self._img_rank = (
            outlier_df
            .groupby(['cobbType', 'image_id'])
            .agg(max_std=('std', 'max'), img_type=('img_type', 'first'))
            .reset_index()
        )

        cobb_types = sorted(outlier_df['cobbType'].unique())

        self._w_cobb = widgets.Dropdown(
            options=cobb_types,
            value='Cobb-regional' if 'Cobb-regional' in cobb_types else cobb_types[0],
            description='cobbType:',
            layout=widgets.Layout(width='340px'),
            style={'description_width': 'initial'},
        )
        self._w_n = widgets.IntSlider(
            value=30, min=5, max=200, step=5,
            description='Top N Bilder:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='350px'),
        )
        self._w_img = widgets.Dropdown(
            options=[],
            description='Bild:',
            layout=widgets.Layout(width='520px'),
            style={'description_width': 'initial'},
        )
        self._out = widgets.Output()

        self._w_cobb.observe(self._refresh_image_list, names='value')
        self._w_n.observe(self._refresh_image_list, names='value')
        self._w_img.observe(self._render, names='value')

        self._refresh_image_list()

    # ------------------------------------------------------------------

    def _refresh_image_list(self, _=None):
        cobb = self._w_cobb.value
        n = self._w_n.value
        ranked = (
            self._img_rank[self._img_rank['cobbType'] == cobb]
            .sort_values('max_std', ascending=False)
            .head(n)
        )
        opts = [
            (f"{r['img_type']:18s}  ID {r['image_id']}  σ_max={r['max_std']:.1f}°", r['image_id'])
            for _, r in ranked.iterrows()
        ]
        self._w_img.unobserve(self._render, names='value')
        self._w_img.options = opts
        self._w_img.observe(self._render, names='value')
        if opts:
            self._render()

    def _load_xray(self, image_id):
        url = self._urls.get(image_id)
        if not url:
            return None
        try:
            resp = requests.get(url, timeout=15)
            resp.raise_for_status()
            return Image.open(io.BytesIO(resp.content)).convert('RGB')
        except Exception as exc:
            print(f'[Image load error] {exc}')
            return None

    def _render(self, _=None):
        image_id = self._w_img.value
        cobb = self._w_cobb.value
        if image_id is None:
            return

        with self._out:
            clear_output(wait=True)

            xray = self._load_xray(image_id)
            measurements = self._df4d[self._df4d['id'] == image_id]

            # Koordinaten-Skalierungsfaktor: Konva-Stage hat feste Höhe 600px.
            # Alle gespeicherten Koordinaten sind in diesem Display-Raum.
            scale = (xray.height / KONVA_STAGE_HEIGHT) if xray else 1.0

            fig, (ax_img, ax_tbl) = plt.subplots(
                1, 2, figsize=(16, 10),
                gridspec_kw={'width_ratios': [2, 1]},
            )

            # --- X-ray anzeigen ---
            if xray:
                ax_img.imshow(xray, origin='upper', aspect='equal')
            else:
                ax_img.set_facecolor('#222')
                ax_img.text(0.5, 0.5, 'Bild nicht verfügbar\n(URL abgelaufen?)',
                            ha='center', va='center', color='white',
                            transform=ax_img.transAxes, fontsize=12)

            # --- Overlay pro Rater ---
            for i, (_, row) in enumerate(measurements.iterrows()):
                xmd = row.get('xray_measurement_data')
                if not isinstance(xmd, dict):
                    continue
                nodes_raw = xmd.get('stateObject', {}).get('nodes', [])
                if len(nodes_raw) < 3:
                    continue

                color = self._COLORS[i % len(self._COLORS)]
                rater = row.get('rater_name', f'Rater {i+1}')

                # Nodes und Spline mit korrigierter Skalierung
                nodes_xy = [(n['x'] * scale, n['y'] * scale) for n in nodes_raw]
                spline = interpolate_bspline(nodes_xy, num_steps=20)

                if spline:
                    ax_img.plot(
                        [p[0] for p in spline], [p[1] for p in spline],
                        '-', color=color, linewidth=2, label=rater, alpha=0.85, zorder=3,
                    )

                ax_img.scatter(
                    [xy[0] for xy in nodes_xy], [xy[1] for xy in nodes_xy],
                    color=color, s=35, zorder=4, edgecolors='white', linewidths=0.5,
                )

                # CobbLines für Cobb-regional / Cobb-segmental
                if cobb in self._COBB_LINE_TYPES:
                    cobb_lines = xmd.get('stateObject', {}).get('cobbLines', [])
                    for cl in cobb_lines:
                        if cl.get('mode') == cobb:
                            ax_img.plot(
                                [cl['x1'] * scale, cl['x2'] * scale],
                                [cl['y1'] * scale, cl['y2'] * scale],
                                '-', color=color, linewidth=2.5, alpha=0.9, zorder=5,
                            )

            ax_img.legend(loc='upper right', fontsize=9, framealpha=0.8)
            img_type = measurements['img_type'].iloc[0] if len(measurements) else ''
            ax_img.set_title(f'ID {image_id}  ({img_type})', fontsize=12)
            ax_img.axis('off')

            # --- Rechte Spalte: Winkelpaare und Einzelmessungen ---
            angle_rows = self._angles[
                (self._angles['image_id'] == image_id) &
                (self._angles['cobbType'] == cobb)
            ][['rater_name', 'vertebra_upper', 'vertebra_lower', 'angle']].copy()
            angle_rows['angle'] = angle_rows['angle'].round(1)
            angle_rows = angle_rows.sort_values(['vertebra_upper', 'rater_name'])

            pair_std = (
                self._outlier[
                    (self._outlier['image_id'] == image_id) &
                    (self._outlier['cobbType'] == cobb)
                ][['vertebra_upper', 'vertebra_lower', 'mean_angle', 'std', 'n_raters']]
                .sort_values('std', ascending=False)
            )

            ax_tbl.axis('off')
            ax_tbl.set_title(f'cobbType: {cobb}', fontsize=11)

            y = 0.98
            lh = 0.045

            def txt(s, bold=False, gray=False):
                nonlocal y
                ax_tbl.text(0.02, y, s, fontsize=9 if not bold else 9,
                            fontweight='bold' if bold else 'normal',
                            fontfamily='monospace',
                            color='gray' if gray else 'black',
                            transform=ax_tbl.transAxes, va='top')
                y -= lh

            if not pair_std.empty:
                txt('Winkelpaare (σ absteigend):', bold=True)
                txt(f"{'Upper':>8}  {'Lower':>6}  {'Mean':>6}  {'σ':>5}  n", gray=True)
                for _, pr in pair_std.iterrows():
                    txt(f"{pr['vertebra_upper']:>8}  {pr['vertebra_lower']:>6}  "
                        f"{pr['mean_angle']:>5.1f}°  {pr['std']:>4.1f}°  {int(pr['n_raters'])}")

            y -= lh
            if not angle_rows.empty:
                txt('Einzelmessungen:', bold=True)
                txt(f"{'Rater':>12}  {'Upper':>8}  {'Lower':>6}  {'°':>6}", gray=True)
                for _, ar in angle_rows.iterrows():
                    txt(f"{str(ar['rater_name']):>12}  {ar['vertebra_upper']:>8}  "
                        f"{ar['vertebra_lower']:>6}  {ar['angle']:>5.1f}°")
            else:
                ax_tbl.text(0.5, 0.5, f'Keine Daten\nfür cobbType={cobb}',
                            ha='center', va='center', transform=ax_tbl.transAxes,
                            fontsize=11, color='gray')

            plt.tight_layout()
            plt.show()

    def show(self):
        display(widgets.VBox([
            widgets.HBox([self._w_cobb, self._w_n]),
            self._w_img,
            self._out,
        ]))

In [4]:
viewer = OutlierViewer(df4d, outlier_df, angles_df, img_urls)
viewer.show()